In [2]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.data.preprocess import PreprocessingConfig
from src.pipeline.data_pipeline import (
    build_nazario_modeling_dataframe,
    print_summary,
    save_dataframe,
)



In [2]:
config = PreprocessingConfig(include_raw_text=False)
records, filtered_records, summary, df = build_nazario_modeling_dataframe(
    Path("../data/raw/nazario").resolve(),
    config=config,
    min_length=10,
)

print_summary(summary)
save_dataframe(df, Path("../data/processed/nazario_clean.csv").resolve())

Total: 4572
Label counts: Counter({'phishing': 4572})
Empty combined text: 45
Empty subject: 45
Empty body: 110
Parse failures: 0
Used fallback: 0
Full fallback: 0
Body fallback: 0
Min length: 12
Max length: 17558
Avg length: 1509.5270598630439


In [3]:
df = pd.read_csv(Path("../data/processed/nazario_clean.csv").resolve())

In [4]:
phishing_example = df[df["label"] == 1].iloc[0]

pd.set_option("display.max_colwidth", None)

print(phishing_example)

text     Billing Issues\n\nDear valued\neBay member:\n\nWe recently have determined that different computers\nhave logged onto your eBay account, and multiple\npassword failures were present before the logons. We\nnow need you to re-confirm your account information to\nus. If this is not completed by \nNovember 30,\n2004\n, we will be forced to suspend your\naccount indefinitely, as it may have been used for\nfraudulent purposes. We thank you for your cooperation\nin this manner.\n\nTo confirm your eBay records click here: \n\nhttp://cgi1.ebay.com/aw-cgi/ebayISAPI.dll?UPdate\n\nWe appreciate your\nsupport and understanding, as we work together to keep\neBay a safe place to trade.\n\nThank you for your patience in this matter.\n\nTrust and Safety\nDepartment\n\neBay Inc.\n\nPlease do\nnot reply to this e-mail as this is only a\nnotification. Mail sent to this address cannot be\nanswered. \n\nCopyright 1995-2004 \neBay\nInc.\n All Rights Reserved. Designated trademarks\nand brands are th

In [5]:
df["text_norm"] = df["text"].str.lower().str.strip()
df = df.drop_duplicates(subset=["text_norm"])
df = df.drop(columns=["text_norm"])

In [6]:
save_dataframe(df, Path("../data/processed/nazario_clean_dedup.csv"))
enron_df = pd.read_csv("../data/processed/enron_clean.csv")

# keep only legit emails
enron_ham_df = enron_df[enron_df["label"] == 0]

nazario_df = pd.read_csv("../data/processed/nazario_clean_dedup.csv")

combined_df = pd.concat([enron_ham_df, nazario_df], ignore_index=True)
combined_df = combined_df.drop_duplicates(subset=["text"])

In [6]:
combined_df["label"].value_counts()

label
0    18263
1     3380
Name: count, dtype: int64

In [7]:
phishing_df = combined_df[combined_df["label"] == 1]
ham_df = combined_df[combined_df["label"] == 0]

ham_sample = ham_df.sample(len(phishing_df), random_state=42)

balanced_df = pd.concat([ham_sample, phishing_df])

In [8]:
balanced_df["label"].value_counts()

label
0    3380
1    3380
Name: count, dtype: int64

In [5]:
enron_df = pd.read_csv("../data/processed/enron_clean.csv")

# keep only legit emails
enron_ham_df = enron_df[enron_df["label"] == 0]

nazario_df = pd.read_csv("../data/processed/nazario_clean_dedup.csv")

combined_df = pd.concat([enron_ham_df, nazario_df], ignore_index=True)
combined_df = combined_df.drop_duplicates(subset=["text"])

In [21]:
df = balanced_df
phishing_examples = df[df["label"] == 0].head(3)

pd.set_option("display.max_colwidth", None)

print(phishing_examples)

In [16]:
df[df["label"] == 1]

text  \
19082                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             Billing Issues\n\nDear valued\neBay member:\n\nWe recently have determined that different computers\nhave logged onto your eBay account, and multiple\npassword failures were present before the logons. We\nnow need you to re-confirm your account information to\nus. If this is not completed by \nNovember 30,\n2004\n, we will be forced to suspend your\naccount indefinitely, as it may have been used for\nfr